In [1]:
import random
import torch
from torch import nn
from torch.utils.data import DataLoader, IterableDataset, get_worker_info
from tqdm import tqdm

from pathlib import Path
import math
from tabulate import tabulate
from collections import deque
from datetime import datetime, timedelta
import numpy as np

from statistics import mean
from types import SimpleNamespace

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
def fit_ea(train_files):
    covariance_sum = None
    count = 0

    for path in train_files:
        with np.load(path, allow_pickle=False) as data:
            signals = data["eeg"]  # (windows, channels, time)

            for signal in signals:
                cov = np.cov(signal.astype(np.float64))

                if covariance_sum is None:
                    covariance_sum = np.zeros_like(cov)

                covariance_sum += cov
                count += 1

    if count == 0:
        raise ValueError("Tập train không có cửa sổ EEG")

    reference = covariance_sum / count

    eigenvalues, eigenvectors = np.linalg.eigh(reference)

    # Chặn trị riêng quá nhỏ để phép nghịch đảo ổn định
    floor = max(eigenvalues.max() * 1e-6, np.finfo(np.float64).tiny)
    eigenvalues = np.maximum(eigenvalues, floor)

    ea_matrix = (
        eigenvectors * eigenvalues**-0.5
    ) @ eigenvectors.T

    return torch.from_numpy(ea_matrix.astype(np.float32))

In [4]:
LEFT_MAT  = [0, 1, 2, 3, 4, 5, 6, 7]
RIGHT_MAT = [12, 13, 14, 15, 8, 9, 10, 11]

def read_labels(path, data=None):
    """Read the per-window labels without loading the EEG array."""
    if data is not None:
        return np.asarray(data["labels"], dtype=np.int64)
    with np.load(path, allow_pickle=False) as archive:
        return np.asarray(archive["labels"], dtype=np.int64)


def apply_flip_augmentation(x, left_mat=LEFT_MAT, right_mat=RIGHT_MAT):
    """Swap the left and right bipolar channels."""
    x_aug = x.clone()
    x_aug[list(left_mat)] = x[list(right_mat)]
    x_aug[list(right_mat)] = x[list(left_mat)]
    return x_aug


def apply_time_warp_augmentation(x, ratio):
    """Stretch a central crop back to the original window length."""
    if not 0 <= ratio < 1:
        raise ValueError("Time-warp ratio must be in [0, 1)")
    n_times = x.shape[-1]
    removed = int(n_times * ratio)
    if removed == 0:
        return x
    first = removed // 2
    cropped = x[:, first:first + n_times - removed]
    return torch.nn.functional.interpolate(
        cropped.unsqueeze(0), size=n_times, mode="linear", align_corners=False
    ).squeeze(0)


In [5]:
class EEGDataset(IterableDataset):
    """
    EEG Dataset hỗ trợ tỷ lệ Preictal : Interictal tùy chỉnh.

    Label:
        0 = Interictal
        1 = Preictal

    balanced=True, class_ratio=(1, 2):
        1 Preictal + 2 Interictal trong mỗi nhóm 3 mẫu.

    Validation/Test:
        balanced=False để giữ phân bố dữ liệu gốc.
    """

    def __init__(
        self,
        files,
        shuffle=False,
        normalize=True,
        augment=False,
        balanced=False,
        class_ratio=(1, 2),
        n_channels=18,
        ea_matrix=None,
        seed=42,
    ):
        self.files = [Path(p) for p in files]
        self.shuffle = shuffle
        self.normalize = normalize
        self.augment = augment
        self.seed = seed
        self.balanced = balanced
        self.epoch = 0
        self.n_channels = n_channels
        self.ea_matrix = ea_matrix

        # class_ratio = (Preictal, Interictal)
        self.class_ratio = tuple(class_ratio)

        if len(self.class_ratio) != 2 or any(
            not isinstance(v, (int, np.integer)) or v <= 0
            for v in self.class_ratio
        ):
            raise ValueError("class_ratio phải gồm 2 số nguyên dương")

        # Thứ tự sinh mẫu: 1 preictal, 2 interictal
        self.pattern = (
            [1] * self.class_ratio[0]
            + [0] * self.class_ratio[1]
        )

        if not self.files:
            raise ValueError("No NPZ files found")

        self.file_counts = {}
        self.class_counts = np.zeros(2, dtype=np.int64)
        self.class_files = {0: [], 1: []}

        for path in self.files:
            labels = read_labels(path)
            self.file_counts[path] = len(labels)

            if self.balanced:
                if labels.ndim != 1 or not np.isin(
                    labels, [0, 1]
                ).all():
                    raise ValueError(
                        f"{path}: balanced mode requires binary labels"
                    )

                counts = np.bincount(labels, minlength=2)
                self.class_counts += counts

                for label in (0, 1):
                    if counts[label]:
                        self.class_files[label].append(path)

        if self.balanced:
            if (self.class_counts == 0).any():
                raise ValueError(
                    "Balanced mode requires both labels 0 and 1"
                )

            # Số nhóm cần tạo để sử dụng hết ít nhất
            # một lớp trước khi lặp dữ liệu.
            n_pre = int(self.class_counts[1])
            n_inter = int(self.class_counts[0])

            pre_per_group = self.class_ratio[0]
            inter_per_group = self.class_ratio[1]

            self.n_groups = max(
                math.ceil(n_pre / pre_per_group),
                math.ceil(n_inter / inter_per_group),
            )

    def __len__(self):
        if self.balanced:
            return self.n_groups * len(self.pattern)

        return sum(
            self.file_counts[path] for path in self.files
        )

    def _windows(self, files, rng, label=None):
        files = list(files)

        if self.shuffle:
            rng.shuffle(files)

        for path in files:
            with np.load(path, allow_pickle=False) as data:
                labels = read_labels(path, data)
                eeg = np.asarray(data["eeg"], dtype=np.float32)

            if (
                eeg.ndim != 3
                or eeg.shape[1] != self.n_channels
                or labels.shape != (len(eeg),)
                or len(eeg) != self.file_counts[path]
            ):
                raise ValueError(
                    f"{path}: expected EEG "
                    f"(windows, {self.n_channels}, samples) "
                    f"and one label per window; "
                    f"got {eeg.shape}, {labels.shape}"
                )

            indices = (
                np.arange(len(eeg))
                if label is None
                else np.flatnonzero(labels == label)
            )

            if self.shuffle:
                rng.shuffle(indices)

            for idx in indices:
                yield eeg[idx], int(labels[idx])

    def _balanced_windows(self, rng):
        streams = {
            label: self._windows(
                self.class_files[label], rng, label
            )
            for label in (0, 1)
        }

        for _ in range(self.n_groups):
            for label in self.pattern:
                try:
                    sample = next(streams[label])

                except StopIteration:
                    # Đọc lại lớp đã hết, shuffle lại nếu bật
                    streams[label] = self._windows(
                        self.class_files[label], rng, label
                    )
                    sample = next(streams[label])

                yield sample

    def __iter__(self):
        worker = get_worker_info()

        if self.balanced and worker is not None:
            raise ValueError(
                "Use num_workers=0 with balanced=True"
            )

        rng = np.random.default_rng(
            self.seed
            + self.epoch
            + (worker.id if worker else 0)
        )

        self.epoch += 1

        files = (
            self.files[worker.id::worker.num_workers]
            if worker else self.files
        )

        windows = (
            self._balanced_windows(rng)
            if self.balanced
            else self._windows(files, rng)
        )

        for signal, label in windows:
            x = torch.from_numpy(signal)

            if self.normalize:
                if self.ea_matrix is None:
                    raise ValueError(
                        "Cần truyền ea_matrix khi normalize=True"
                    )

                x = self.ea_matrix @ x

                mean = x.mean(dim=-1, keepdim=True)
                std = x.std(
                    dim=-1, keepdim=True
                ).clamp_min(1e-8)

                x = (x - mean) / std

            if self.augment:
                if rng.random() < 0.2:
                    x = apply_flip_augmentation(x)

                if rng.random() < 0.2:
                    x = apply_time_warp_augmentation(
                        x, rng.uniform(0.05, 0.25)
                    )

            yield x, torch.tensor(
                label, dtype=torch.long
            )


In [6]:
'''
=================================================
coding:utf-8
@Time:      2025/4/13 17:06
@File:      DBConformer.py
@Author:    Ziwei Wang
@Function:  Original model code of "DBConformer: Dual-Branch Convolutional Transformer for EEG Decoding"
=================================================
'''

import math
from timm.models.layers import trunc_normal_
import torch.nn.functional as F
import torch
from torch import nn
from torch import Tensor
from einops import rearrange
from torch.backends import cudnn
cudnn.benchmark = False
cudnn.deterministic = True


class Conv(nn.Module):
    def __init__(self, conv, activation=None, bn=None):
        nn.Module.__init__(self)
        self.conv = conv
        self.activation = activation
        if bn:
            self.conv.bias = None
        self.bn = bn

    def forward(self, x):
        x = self.conv(x)
        if self.bn:
            x = self.bn(x)
        if self.activation:
            x = self.activation(x)
        return x


class InterFre(nn.Module):
    def __init__(self):
        nn.Module.__init__(self)

    def forward(self, x):
        out = sum(x)
        out = F.gelu(out)
        return out


class Stem(nn.Module):
    def __init__(self, data_name, in_planes, out_planes = 64, kernel_size=63, patch_size=125, radix=2):
        nn.Module.__init__(self)
        self.in_planes = in_planes
        self.out_planes = out_planes
        self.mid_planes = out_planes * radix
        self.kernel_size = kernel_size
        self.radix = radix
        self.data_name = data_name

        self.sconv = Conv(nn.Conv1d(self.in_planes, self.mid_planes, 1, bias=False, groups = radix),
                          bn=nn.BatchNorm1d(self.mid_planes), activation=None)

        self.tconv = nn.ModuleList()
        for _ in range(self.radix):
            self.tconv.append(Conv(nn.Conv1d(self.out_planes, self.out_planes, kernel_size, 1, groups=self.out_planes, padding=kernel_size // 2, bias=False,),
                                   bn=nn.BatchNorm1d(self.out_planes), activation=None))
            kernel_size //= 2

        self.interFre = InterFre()

        self.downSampling = nn.AvgPool1d(patch_size, patch_size)
        self.dp = nn.Dropout(0.5)  #

    def forward(self, x):
        N, C, T = x.shape
        out = self.sconv(x)
        out = torch.split(out, self.out_planes, dim=1)
        out = [m(x) for x, m in zip(out, self.tconv)]
        out = self.interFre(out)
        if self.data_name != 'MI1-7':  # MI1-7 is the Blankertz2007 dataset in our paper
            out = out[:, :, :-1]  # Example: 14001 has 1001 time points, we exclude the final point
        out = self.downSampling(out)
        out = self.dp(out)
        return out


class PatchEmbeddingTemporal(nn.Module):
    def __init__(self, data_name, in_planes, out_planes, kernel_size, radix, patch_size, time_points, num_classes):
        '''
        Outputs patch embeddings of shape (B, P, D)
        '''
        super().__init__()
        self.data_name = data_name
        self.stem = Stem(
            data_name=self.data_name,
            in_planes=in_planes * radix,
            out_planes=out_planes,
            kernel_size=kernel_size,
            patch_size=patch_size,
            radix=radix
        )
        self.apply(self.initParms)

    def initParms(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.01)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, (nn.LayerNorm, nn.BatchNorm1d, nn.BatchNorm2d)):
            if m.weight is not None:
                nn.init.constant_(m.weight, 1.0)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, (nn.Conv1d, nn.Conv2d)):
            trunc_normal_(m.weight, std=.01)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def forward(self, x):  # x: (B, C, T)
        out = self.stem(x)         # (B, D, P)
        out = out.permute(0, 2, 1) # → (B, P, D)
        return out


class CrossAttention(nn.Module):
    def __init__(self, emb_size, num_heads=4, dropout=0.2):
        super().__init__()
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.query_proj = nn.Linear(emb_size, emb_size)
        self.key_proj = nn.Linear(emb_size, emb_size)
        self.value_proj = nn.Linear(emb_size, emb_size)
        self.out_proj = nn.Linear(emb_size, emb_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value):  # query: (B, P1, D), key_value: (B, P2, D)
        Q = self.query_proj(query)
        K = self.key_proj(key_value)
        V = self.value_proj(key_value)

        Q = rearrange(Q, 'b n (h d) -> b h n d', h=self.num_heads)
        K = rearrange(K, 'b n (h d) -> b h n d', h=self.num_heads)
        V = rearrange(V, 'b n (h d) -> b h n d', h=self.num_heads)

        attn_scores = torch.einsum('bhqd, bhkd -> bhqk', Q, K) / (self.emb_size ** 0.5)
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.dropout(attn_probs)

        out = torch.einsum('bhqk, bhvd -> bhqd', attn_probs, V)
        out = rearrange(out, 'b h n d -> b n (h d)')
        out = self.out_proj(out)

        return out


class SEBlock(nn.Module):
    """bad performance"""
    def __init__(self, in_channels, reduction=16):
        super(SEBlock, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction),
            nn.ReLU(),
            nn.Linear(in_channels // reduction, in_channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        B, C, D = x.shape
        y = x.mean(dim=-1)  # (B, C)
        y = self.fc(y).unsqueeze(-1)  # (B, C, 1)
        return x * y


class PatchEmbeddingSpatial(nn.Module):
    def __init__(self, spa_dim, emb_size=40):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1, spa_dim, kernel_size=25, stride=5, padding=12),  # Output: (B*C, spa_dim, T')
            nn.ELU(),
            nn.AdaptiveAvgPool1d(1),  # Output: (B*C, spa_dim, 1)
            nn.Flatten(),             # → (B*C, 16)
            nn.Linear(spa_dim, emb_size)   # → (B*C, emb_size)
        )

    def forward(self, x):  # x: (B, C, T)
        B, C, T = x.shape
        x = x.unsqueeze(2)         # (B, C, 1, T)
        x = x.reshape(B * C, 1, T)  # → (B*C, 1, T)
        x = self.encoder(x)        # → (B*C, emb_size)
        x = x.view(B, C, -1)       # → (B, C, emb_size)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_size, num_heads, dropout):
        super().__init__()
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.keys = nn.Linear(emb_size, emb_size)
        self.queries = nn.Linear(emb_size, emb_size)
        self.values = nn.Linear(emb_size, emb_size)
        self.att_drop = nn.Dropout(dropout)
        self.projection = nn.Linear(emb_size, emb_size)

    def forward(self, x: Tensor, mask: Tensor = None) -> Tensor:
        queries = rearrange(self.queries(x), "b n (h d) -> b h n d", h=self.num_heads)
        keys = rearrange(self.keys(x), "b n (h d) -> b h n d", h=self.num_heads)
        values = rearrange(self.values(x), "b n (h d) -> b h n d", h=self.num_heads)
        energy = torch.einsum('bhqd, bhkd -> bhqk', queries, keys)
        if mask is not None:
            fill_value = torch.finfo(torch.float32).min
            energy.mask_fill(~mask, fill_value)

        scaling = self.emb_size ** (1 / 2)
        att = F.softmax(energy / scaling, dim=-1)
        att = self.att_drop(att)
        out = torch.einsum('bhal, bhlv -> bhav ', att, values)
        out = rearrange(out, "b h n d -> b n (h d)")
        out = self.projection(out)
        return out


class ResidualAdd(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        res = x
        x = self.fn(x, **kwargs)
        x += res
        return x


class FeedForwardBlock(nn.Sequential):
    def __init__(self, emb_size, expansion, drop_p):
        super().__init__(
            nn.Linear(emb_size, expansion * emb_size),
            nn.GELU(),
            nn.Dropout(drop_p),
            nn.Linear(expansion * emb_size, emb_size),
        )


class GELU(nn.Module):
    def forward(self, input: Tensor) -> Tensor:
        return input * 0.5 * (1.0 + torch.erf(input / math.sqrt(2.0)))


class TransformerEncoderBlock(nn.Sequential):
    def __init__(self,
                 emb_size,
                 num_heads=10,
                 drop_p=0.5,
                 forward_expansion=4,
                 forward_drop_p=0.5):
        super().__init__(
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                MultiHeadAttention(emb_size, num_heads, drop_p),
                nn.Dropout(drop_p)
            )),
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                FeedForwardBlock(
                    emb_size, expansion=forward_expansion, drop_p=forward_drop_p),
                nn.Dropout(drop_p)
            )
            ))


class TransformerEncoder(nn.Sequential):
    def __init__(self, depth, emb_size):
        super().__init__(*[TransformerEncoderBlock(emb_size) for _ in range(depth)])


class ClassificationHead(nn.Sequential):
    def __init__(self, emb_size, n_classes):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(emb_size, 64),
            nn.ELU(),
            nn.Dropout(0.5),
            nn.Linear(64, 32),
            nn.ELU(),
            nn.Dropout(0.3),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        x = x.contiguous().view(x.size(0), -1)
        out = self.fc(x)
        return x, out

class Gate_FC(nn.Sequential):
    def __init__(self, emb_size):
        super().__init__()
        self.fc = nn.Linear(emb_size * 2, emb_size)

    def forward(self, x):
        x = x.contiguous().view(x.size(0), -1)
        out = self.fc(x)
        return out


class DBConformer(nn.Module):
    def __init__(self, args, emb_size=40, tem_depth=5, chn_depth=5, chn=22, n_classes=2):
        super().__init__()

        self.embedding = PatchEmbeddingTemporal(
            data_name=args.data_name,
            in_planes=args.chn,  # number of channels
            out_planes=emb_size,  # Default 40
            kernel_size=63,
            radix=1,
            patch_size=args.patch_size,  # needs to be divisible by the number of time points
            time_points=args.time_sample_num,  # number of time points
            num_classes=args.class_num  # number of classes
        )
        self.channel_embedding = PatchEmbeddingSpatial(spa_dim=args.spa_dim, emb_size=emb_size)  # Default 16
        self.P = args.time_sample_num // args.patch_size  # Example: 1000 // 125 = 8
        self.C = args.chn  # number of channels
        self.D = emb_size
        self.gate_flag = args.gate_flag  # Default False, due to the reduced performance
        self.posemb_flag = args.posemb_flag  # Default True
        self.branch = args.branch  # Default 'all', options=[all, temporal]
        self.chn_atten_flag = args.chn_atten_flag  # Default True

        if args.posemb_flag:
            self.pos_embedding_temporal = nn.Parameter(torch.randn(1, self.P, self.D))
            self.pos_embedding_spatial = nn.Parameter(torch.randn(1, self.C, self.D))

        self.temporal_transformer = TransformerEncoder(tem_depth, emb_size)
        self.spatial_transformer = TransformerEncoder(chn_depth, emb_size)
        # self.cross_attn = CrossAttention(emb_size=emb_size, num_heads=4)  # cross-attention (bad performance)
        if args.gate_flag or self.branch == 'temporal' or self.branch == 'spatial':
            self.gate_fc = Gate_FC(emb_size)  # dual-branch weighting aggregation
            self.classifier = ClassificationHead(emb_size, n_classes)
        else:
            self.classifier = ClassificationHead(emb_size * 2, n_classes)

            if args.chn_atten_flag:
                self.spatial_attn_pool = nn.Sequential(
                    nn.Linear(emb_size, emb_size),  # D → D
                    nn.Tanh(),
                    nn.Linear(emb_size, 1),  # D → 1 (score per channel)
                )
    def forward(self, x):  # x: (B, 1, C, T)
        x = x.squeeze(1)  # → (B, C, T)
        x_embed = self.embedding(x)  # → (B, P, D)
        x_embed_spatial = self.channel_embedding(x)  # (B, C, D)
        if self.posemb_flag:
            x_embed = x_embed + self.pos_embedding_temporal  # temporal positional encoding
            x_embed_spatial = x_embed_spatial + self.pos_embedding_spatial  # spatial positional encoding

        # Temporal Transformer (attention over time dimension)
        x_temporal = self.temporal_transformer(x_embed)  # (B, P, D)
        # Spatial Transformer (attention over channels interpreted as tokens)
        x_spatial = self.spatial_transformer(x_embed_spatial)  # (B, C, D)

        if self.branch == 'temporal':
            x_fused = x_temporal.mean(dim=1)
            _, out = self.classifier(x_fused)  # out: (B, n_classes)
        elif self.branch == 'spatial':  # Using S-Conformer-only doesn't work
            x_fused = x_spatial.mean(dim=1)
            _, out = self.classifier(x_fused)  # out: (B, n_classes)
        elif self.branch == 'all':
            if self.gate_flag:
                # gated-fusion
                gate = torch.sigmoid(
                    self.gate_fc(torch.cat([x_temporal.mean(dim=1), x_spatial.mean(dim=1)], dim=-1)))  # shape: (B, D)
                x_fused = gate * x_spatial.mean(dim=1) + (1 - gate) * x_temporal.mean(dim=1)
            else:
                if self.chn_atten_flag:
                    # Attention Scores
                    x_t = x_temporal.mean(dim=1)
                    attn_scores = self.spatial_attn_pool(x_spatial)  # (B, C, 1)
                    attn_weights = torch.softmax(attn_scores, dim=1)  # (B, C, 1)
                    x_s = torch.sum(attn_weights * x_spatial, dim=1)  # (B, D)
                    x_fused = torch.cat([x_t, x_s], dim=-1)  # → (B, 2*D)
                else:
                    # Mean pooling
                    x_fused = torch.cat([
                        x_temporal.mean(dim=1),
                        x_spatial.mean(dim=1)
                    ], dim=-1)  # → (B, 2*D)
            _, out = self.classifier(x_fused)  # out: (B, n_classes)
        return x_fused, out

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [7]:
def run_epoch(model, loader, criterion, device, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_samples = 0
    confusion = torch.zeros(2, 2, dtype=torch.long, device=device)

    progress = tqdm(
        loader,
        total=len(loader),
        desc="Train" if training else "Validation",
        leave=False,
    )

    with torch.set_grad_enabled(training):
        for x, y in progress:
            x = x.to(device, dtype=torch.float32)
            y = y.to(device, dtype=torch.long).reshape(-1)

            if x.ndim == 3:
                x = x.unsqueeze(1)

            if training:
                optimizer.zero_grad(set_to_none=True)

            _, logits = model(x)

            # criterion = CrossEntropyLoss(reduction="mean")
            loss = criterion(logits, y)

            if not torch.isfinite(loss).item():
                raise RuntimeError("Loss là NaN hoặc infinity.")

            if training:
                loss.backward()
                optimizer.step()

            # Đổi mean loss của batch thành tổng loss,
            # rồi chia tổng số mẫu ở cuối epoch.
            batch_size = y.numel()
            total_loss += loss.detach().item() * batch_size
            total_samples += batch_size

            predictions = logits.detach().argmax(dim=1)
            confusion += torch.bincount(
                y * 2 + predictions,
                minlength=4,
            ).reshape(2, 2)

            progress.set_postfix(
                loss=f"{total_loss / total_samples:.6f}",
                pre=f"{(y == 1).sum().item()}/{batch_size}",
            )

    if total_samples == 0:
        raise ValueError("Loader không có dữ liệu.")

    tn, fp, fn, tp = confusion.cpu().flatten().tolist()

    sensitivity = tp / (tp + fn) if tp + fn else float("nan")
    specificity = tn / (tn + fp) if tn + fp else float("nan")

    return {
        "loss": total_loss / total_samples,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "balanced_accuracy": (sensitivity + specificity) / 2,
        "accuracy": (tp + tn) / (tp + tn + fp + fn),
    }

In [8]:
@torch.inference_mode()
def run_validation(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    tn = fp = fn = tp = 0
    progress = tqdm(
        loader,
        total=len(loader),
        desc="Validation",
        leave=False,
    )
    for x, y in progress:
        x = x.to(device)
        y = y.to(device, dtype=torch.long)

        _, logits = model(x)
        loss = criterion(logits, y)

        if not torch.isfinite(loss).item():
            raise RuntimeError("Validation loss là NaN hoặc infinity.")

        predicted = logits.argmax(dim=1)

        total_loss += loss.item() * y.size(0)
        total_samples += y.size(0)

        tn += ((y == 0) & (predicted == 0)).sum().item()
        fp += ((y == 0) & (predicted == 1)).sum().item()
        fn += ((y == 1) & (predicted == 0)).sum().item()
        tp += ((y == 1) & (predicted == 1)).sum().item()

    if total_samples == 0:
        raise ValueError("Validation loader không có dữ liệu.")

    sensitivity = tp / (tp + fn) if tp + fn else float("nan")
    specificity = tn / (tn + fp) if tn + fp else float("nan")

    return {
        "loss": total_loss / total_samples,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "balanced_accuracy": (sensitivity + specificity) / 2,
    }

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

DEVICE: cuda


In [10]:
import csv
from pathlib import Path, PurePosixPath


def available_folds(csv_path, patient):
    with Path(csv_path).open(encoding="utf-8-sig", newline="") as stream:
        return sorted({int(r["fold"]) for r in csv.DictReader(stream) if r["patient"] == patient})


def load_fold(csv_path, data_root, patient, fold=1, verbose=True):
    """Đọc cả inter/pre theo subset trong CSV; trả về train_files, val_files.

    data_root trỏ vào segment_train. CSV đã chia inter theo EDF cho từng fold,
    hàm này chỉ đọc lại, không chia lại hay cân bằng số cửa sổ hai lớp.
    """
    if isinstance(fold, bool) or not isinstance(fold, int) or fold < 1:
        raise ValueError("fold must be a positive integer, starting at 1")
    root = Path(data_root).resolve()
    files, edfs, clusters, counts = {}, {}, {}, {}
    for subset in ("train", "val"):
        files[subset], edfs[subset], clusters[subset], counts[subset] = [], set(), set(), {0: 0, 1: 0}
    with Path(csv_path).open(encoding="utf-8-sig", newline="") as stream:
        for row in csv.DictReader(stream):
            if row["patient"] != patient or int(row["fold"]) != fold:
                continue
            subset, label = row["subset"], int(row["label"])
            windows = int(row["windows"])
            if subset not in files or label not in (0, 1) or windows <= 0:
                raise ValueError(f"Invalid CSV row: {row}")
            relative = PurePosixPath(row["relative_path"])
            # Nhãn 0: lấy file inter; nhãn 1: lấy file pre.
            kind = "pre" if label == 1 else "inter"
            expected = PurePosixPath(patient) / kind / f"{Path(row['source_edf']).stem}.npz"
            if relative != expected:
                raise ValueError(f"Unexpected relative_path: {relative}")
            path = root.joinpath(*relative.parts).resolve()
            if root not in path.parents:
                raise ValueError(f"Path outside data_root: {path}")
            if not path.is_file():
                raise FileNotFoundError(path)
            files[subset].append(path)  # Cả inter và pre đều vào train/val theo CSV.
            edfs[subset].add(row["source_edf"])
            clusters[subset].update(c for c in row["cluster_ids"].split(";") if c)
            counts[subset][label] += windows
    for subset in files:
        if not files[subset] or len(files[subset]) != len(set(files[subset])):
            raise ValueError(f"{patient}/fold_{fold}: missing or duplicate {subset} files")
        if not all(counts[subset].values()):
            raise ValueError(f"{patient}/fold_{fold}: {subset} requires both labels")
    if edfs["train"] & edfs["val"] or clusters["train"] & clusters["val"]:
        raise ValueError("EDF or seizure cluster appears in both train and val")
    if verbose:
        for subset in ("train", "val"):
            n_inter = sum(path.parent.name == "inter" for path in files[subset])
            n_pre = len(files[subset]) - n_inter
            print(f"{patient}/fold_{fold}/{subset}: "
                  f"inter={n_inter} files, {counts[subset][0]} windows; "
                  f"pre={n_pre} files, {counts[subset][1]} windows")
    return files["train"], files["val"]


def split_train_val(
    patient_dir,
    csv_path: str | Path,
    fold_index: int = 0,
):
    """Trả về train_files, val_files để truyền vào EEGDataset."""
    if isinstance(fold_index, bool) or not isinstance(fold_index, int) or fold_index < 0:
        raise ValueError("fold_index phải là số nguyên >= 0")
    patient_dir = Path(patient_dir)
    return load_fold(csv_path, patient_dir.parent, patient_dir.name, fold=fold_index + 1)

In [11]:
pre_time = "30m"
CSV_PATH = Path(f"/kaggle/input/datasets/nguyentduc/eeg-chb-mit-preprocessig-4-hours/chb_preprocessing_dataset/segment_train_folds/kfold_4h_{pre_time}.csv")

DATA_ROOT = Path(f"/kaggle/input/datasets/nguyentduc/eeg-chb-mit-preprocessig-4-hours/chb_preprocessing_dataset/segment_train_4h_{pre_time}")
# DATA_ROOT = Path(f"/kaggle/input/datasets/nguyentduc/chb-mit-preprocessing-dataset-window-size30/segment_train_4h_{pre_time}")

In [12]:
PATIENT = "chb01"

In [13]:
for f_idx in available_folds(CSV_PATH, PATIENT):
    train_files, val_files = split_train_val(
        patient_dir=DATA_ROOT / PATIENT,
        csv_path=CSV_PATH,
        fold_index=f_idx - 1,
    )
    print(f"Fold {f_idx}: {len(train_files)} train, {len(val_files)} val")

chb01/fold_1/train: inter=3 files, 1873 windows; pre=6 files, 1283 windows
chb01/fold_1/val: inter=1 files, 900 windows; pre=1 files, 450 windows
Fold 1: 9 train, 2 val
chb01/fold_2/train: inter=3 files, 1873 windows; pre=5 files, 1285 windows
chb01/fold_2/val: inter=1 files, 900 windows; pre=2 files, 448 windows
Fold 2: 8 train, 3 val
chb01/fold_3/train: inter=3 files, 2240 windows; pre=5 files, 1285 windows
chb01/fold_3/val: inter=1 files, 533 windows; pre=2 files, 448 windows
Fold 3: 8 train, 3 val
chb01/fold_4/train: inter=3 files, 2333 windows; pre=5 files, 1346 windows
chb01/fold_4/val: inter=1 files, 440 windows; pre=2 files, 387 windows
Fold 4: 8 train, 3 val


In [14]:
pateint_dir   = DATA_ROOT / PATIENT
pateint_files = [f for f in pateint_dir.rglob("*.npz")]

ea_matrix = fit_ea(pateint_files)
print("EA Shape: ", ea_matrix.shape)
print("min     : ", ea_matrix.min())
print("max     : ", ea_matrix.max())

EA Shape:  torch.Size([18, 18])
min     :  tensor(-378066.6562)
max     :  tensor(414153.8750)


In [15]:
# CHECKPOINT_DIR = Path(f"/kaggle/working/no_pretrain{PATIENT}_30w")
# CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = Path(f"/kaggle/working/no_pretrain{PATIENT}")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [16]:
# for f_idx in available_folds(CSV_PATH, PATIENT):
#     print(f"\n==================== Fold {f_idx} ========================\n")
    
#     # BEST_CHECKPOINT = CHECKPOINT_DIR / f"best_model_fold_{f_idx}_{pre_time}.pt"
#     # LAST_CHECKPOINT = CHECKPOINT_DIR / f"last_model_fold_{f_idx}_{pre_time}.pt"

#     BEST_CHECKPOINT = CHECKPOINT_DIR / f"best_model_fold_{f_idx}.pt"
#     LAST_CHECKPOINT = CHECKPOINT_DIR / f"last_model_fold_{f_idx}.pt"
    
#     args = SimpleNamespace(
#         data_name="MI1-7",
#         chn=18,
#         time_sample_num=4*256,
#         class_num=2,
#         patch_size=128,
#         spa_dim=16,
#         posemb_flag=True,
#         chn_atten_flag=True,
#         gate_flag=False,
#         branch="all",
#     )
    
#     model = DBConformer(
#         args,
#         emb_size=40,
#         tem_depth=2,
#         chn_depth=2,
#         chn=args.chn,
#         n_classes=args.class_num,
#     ).to(device)

    
#     # checkpoint = torch.load("/kaggle/input/datasets/nguyentduc/eeg-chb-mit-weight-pretraining/best_pretrain_model.pt",
#     #                     map_location=device,
#     #                     weights_only = True)
#     # model.load_state_dict(checkpoint["model_state_dict"])
#     # model.to(device)

    
#     EPOCHS = 200
#     PATIENCE = 20
#     MIN_DELTA = 1e-3
#     optimizer = torch.optim.AdamW(model.parameters(),
#                                   lr=1e-4,
#                                   weight_decay=1e-4,)
    
#     criterion = nn.CrossEntropyLoss(reduction="mean")
    
#     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#         optimizer,
#         mode="min",
#         factor=0.5,
#         patience=5,
#         threshold=MIN_DELTA,
#         threshold_mode="abs",
#         min_lr=1e-6,)

#     # Theo dõi checkpoint tốt nhất.
#     best_bal_acc = float("-inf")
#     best_model_val_loss = float("inf")
#     best_epoch = None
    
#     # Theo dõi cải thiện đủ lớn cho early stopping.
#     early_stop_best = float("inf")
#     epochs_without_improvement = 0
    
#     # History training
#     loss_hist = []
#     sen_hist = []
#     spec_hist = []
#     bal_acc_hist = []
    
#     # History validation
#     val_loss_hist = []
#     val_sen_hist = []
#     val_spec_hist = []
#     val_bal_acc_hist = []
    
#     headers = [
#         "Tập",
#         "Loss",
#         "Sen (%)",
#         "Spec (%)",
#         "Bal_acc (%)",
#     ]
#     train_files, val_files = split_train_val(patient_dir=DATA_ROOT / PATIENT,
#                                              csv_path=CSV_PATH,
#                                              fold_index=f_idx - 1,)
    
#     train_set = EEGDataset(train_files,
#                            shuffle=True,
#                            normalize=True,
#                            augment=False,
#                            n_channels=18,
#                            balanced=True,
#                            class_ratio=(1, 1),
#                            ea_matrix=ea_matrix,
#                            seed=SEED,)

#     val_set = EEGDataset(val_files,
#                            shuffle=False,
#                            normalize=True,
#                            augment=False,
#                            n_channels=18,
#                            balanced=False,
#                            ea_matrix=ea_matrix,
#                            seed=SEED,)

#     train_loader = DataLoader(train_set,
#                               batch_size=64,
#                               num_workers=0,)
    
#     val_loader = DataLoader(val_set,
#                               batch_size=64,
#                               num_workers=0,)

#     for epoch in range(1, EPOCHS + 1):
#         print(f"\nEpoch {epoch}/{EPOCHS}")
    
#         model.train()
    
#         train = run_epoch(
#             model,
#             train_loader,
#             criterion,
#             device,
#             optimizer,
#         )
    
#         val = run_validation(
#             model,
#             val_loader,
#             criterion,
#             device,
#         )
    
#         train_loss = float(train["loss"])
#         val_loss = float(val["loss"])
#         val_bal_acc = float(val["balanced_accuracy"])
    
#         if not math.isfinite(train_loss):
#             raise RuntimeError("Training loss là NaN hoặc infinity.")
    
#         if not math.isfinite(val_loss):
#             raise RuntimeError("Validation loss là NaN hoặc infinity.")
    
#         if not math.isfinite(val_bal_acc):
#             raise RuntimeError(
#                 "Validation balanced accuracy không hợp lệ. "
#                 "Kiểm tra validation có đủ hai lớp."
#             )
    
#         loss_hist.append(train_loss)
#         sen_hist.append(float(train["sensitivity"]))
#         spec_hist.append(float(train["specificity"]))
#         bal_acc_hist.append(float(train["balanced_accuracy"]))
    
#         val_loss_hist.append(val_loss)
#         val_sen_hist.append(float(val["sensitivity"]))
#         val_spec_hist.append(float(val["specificity"]))
#         val_bal_acc_hist.append(val_bal_acc)
    
#         # Scheduler theo validation Balanced Accuracy
#         old_lr = optimizer.param_groups[0]["lr"]
#         scheduler.step(val_loss)
#         next_lr = optimizer.param_groups[0]["lr"]
    
#         if next_lr < old_lr:
#             print(f"Giảm LR: {old_lr:.2e} → {next_lr:.2e}")
    
#         # Chọn best theo validation balanced accuracy.
#         # Nếu bằng nhau, ưu tiên validation loss thấp hơn.
#         saved = (
#             val_bal_acc > best_bal_acc
#             or (
#                 val_bal_acc == best_bal_acc
#                 and val_loss < best_model_val_loss
#             )
#         )
    
#         if saved:
#             best_bal_acc = val_bal_acc
#             best_model_val_loss = val_loss
#             best_epoch = epoch
    
#         # Early stopping vẫn theo validation loss.
#         if val_loss < early_stop_best - MIN_DELTA:
#             early_stop_best = val_loss
#             epochs_without_improvement = 0
#         else:
#             epochs_without_improvement += 1
    
#         stopping = epochs_without_improvement >= PATIENCE
    
#         checkpoint = {
#             "epoch": epoch,
#             "model_state_dict": model.state_dict(),
#             "optimizer_state_dict": optimizer.state_dict(),
#             "scheduler_state_dict": scheduler.state_dict(),
    
#             "monitor": "val_balanced_accuracy",
#             "early_stop_monitor": "val_loss",
#             "train_loss": train_loss,
#             "val_loss": val_loss,
#             "best_bal_acc": best_bal_acc,
#             "best_model_val_loss": best_model_val_loss,
#             "best_epoch": best_epoch,
#             "early_stop_best": early_stop_best,
#             "epochs_without_improvement": epochs_without_improvement,
    
#             "train_metrics": {
#                 key: float(value) for key, value in train.items()
#             },
#             "val_metrics": {
#                 key: float(value) for key, value in val.items()
#             },
    
#             "loss_hist": loss_hist.copy(),
#             "sen_hist": sen_hist.copy(),
#             "spec_hist": spec_hist.copy(),
#             "bal_acc_hist": bal_acc_hist.copy(),
    
#             "val_loss_hist": val_loss_hist.copy(),
#             "val_sen_hist": val_sen_hist.copy(),
#             "val_spec_hist": val_spec_hist.copy(),
#             "val_bal_acc_hist": val_bal_acc_hist.copy(),
#         }
    
#         torch.save(checkpoint, LAST_CHECKPOINT)
    
#         if saved:
#             torch.save(checkpoint, BEST_CHECKPOINT)
    
#         status = (
#             "Saved best; Early stop" if saved and stopping
#             else "Early stop" if stopping
#             else "Saved best" if saved
#             else "Continue"
#         )
    
#         rows = [
#             [
#                 "Train",
#                 train_loss,
#                 train["sensitivity"] * 100,
#                 train["specificity"] * 100,
#                 train["balanced_accuracy"] * 100,
#             ],
#             [
#                 "Val",
#                 val_loss,
#                 val["sensitivity"] * 100,
#                 val["specificity"] * 100,
#                 val_bal_acc * 100,
#             ],
#         ]
    
#         print(
#             tabulate(
#                 rows,
#                 headers=headers,
#                 tablefmt="plain",
#                 floatfmt=("", ".6f", ".2f", ".2f", ".2f"),
#                 numalign="right",
#                 stralign="left",
#             )
#         )
    
#         print(
#             f"Best val Bal_acc: {best_bal_acc * 100:.2f}% "
#             f"(epoch {best_epoch}) | "
#             f"Next LR: {next_lr:.2e} | "
#             f"Patience: {epochs_without_improvement}/{PATIENCE} | "
#             f"Status: {status}"
#         )
    
#         if stopping:
#             print(
#                 f"Dừng training: {PATIENCE} epoch liên tiếp "
#                 "không cải thiện đủ validation loss."
#             )
#             break
    
#     print(f"\nBest validation Bal_acc: {best_bal_acc * 100:.2f}%")
#     print(f"Validation loss tại best checkpoint: {best_model_val_loss:.6f}")
#     print(f"Best epoch: {best_epoch}")
#     print(f"Best checkpoint: {BEST_CHECKPOINT}")
#     print(f"Last checkpoint: {LAST_CHECKPOINT}")

In [17]:
# from pathlib import Path
# from zipfile import ZipFile, ZIP_DEFLATED
# from IPython.display import FileLink, display

# working_dir = Path("/kaggle/working")
# zip_path = working_dir / "all_folders.zip"

# folders = [p for p in working_dir.iterdir() if p.is_dir()]

# with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
#     for folder in folders:
#         print(f"Đang nén: {folder.name}")
#         archive.write(folder, folder.relative_to(working_dir))

#         for path in folder.rglob("*"):
#             archive.write(path, path.relative_to(working_dir))

# print(f"Đã nén {len(folders)} thư mục → {zip_path}")
# display(FileLink(str(zip_path)))

In [23]:
# TEST_DIR = Path(
#     f"/kaggle/input/datasets/nguyentduc/chb-mit-test-dataset/test_continuous/{PATIENT}"
# )

# BEST_CHECKPOINT = f"/kaggle/input/datasets/nguyentduc/eeg-chb-mit-weight-4h-30m/weight_4h_30m/best_model_{PATIENT}_val.pt"

# TEST_DIR = Path(
#     f"/kaggle/input/datasets/nguyentduc/eeg-chb-mit-preprocessig-4-hours/chb_preprocessing_dataset/test_continuous/{PATIENT}"
# )

TEST_DIR = Path(
    f"/kaggle/input/datasets/nguyentduc/eeg-chb-mit-preprocessig-4-hours/chb_preprocessing_dataset/val_continuous/{PATIENT}"
)

BEST_CHECKPOINT = f"/kaggle/working/chb01/best_model_fold_1_30m.pt"


checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=device,
    weights_only=True,
)

args = SimpleNamespace(
    data_name="MI1-7",
    chn=18,
    time_sample_num=4*256,
    class_num=2,
    patch_size=128,
    spa_dim=16,
    posemb_flag=True,
    chn_atten_flag=True,
    gate_flag=False,
    branch="all",
)

model = DBConformer(
    args,
    emb_size=40,
    tem_depth=2,
    chn_depth=2,
    chn=args.chn,
    n_classes=args.class_num,
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

DBConformer(
  (embedding): PatchEmbeddingTemporal(
    (stem): Stem(
      (sconv): Conv(
        (conv): Conv1d(18, 40, kernel_size=(1,), stride=(1,), bias=False)
        (bn): BatchNorm1d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (tconv): ModuleList(
        (0): Conv(
          (conv): Conv1d(40, 40, kernel_size=(63,), stride=(1,), padding=(31,), groups=40, bias=False)
          (bn): BatchNorm1d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (interFre): InterFre()
      (downSampling): AvgPool1d(kernel_size=(128,), stride=(128,), padding=(0,))
      (dp): Dropout(p=0.5, inplace=False)
    )
  )
  (channel_embedding): PatchEmbeddingSpatial(
    (encoder): Sequential(
      (0): Conv1d(1, 16, kernel_size=(25,), stride=(5,), padding=(12,))
      (1): ELU(alpha=1.0)
      (2): AdaptiveAvgPool1d(output_size=1)
      (3): Flatten(start_dim=1, end_dim=-1)
      (4): Linear(in_features=16, out_features=40

In [24]:
from collections import deque
from datetime import datetime, timedelta
from pathlib import Path
from statistics import mean
import numpy as np
from tabulate import tabulate
import torch


def normalize_function(x, ea_matrix=None):
    # x: (batch, channels, samples)
    if ea_matrix is not None:
        ea_matrix = ea_matrix.to(device=x.device, dtype=x.dtype)
        x = ea_matrix @ x

    mean_val = x.mean(dim=-1, keepdim=True)
    std_val = x.std(dim=-1, keepdim=True).clamp_min(1e-8)

    return (x - mean_val) / std_val


def read_datetime(value):
    return datetime.fromisoformat(str(value.item()))


def format_hms(seconds):
    """Định dạng thời lượng (giây) sang chuỗi hh:mm:ss."""
    if seconds is None:
        return "N/A"
    total_seconds = int(round(max(0.0, float(seconds))))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"


def format_time(dt):
    """Định dạng mốc thời gian datetime sang chuỗi hh:mm:ss."""
    return dt.strftime("%H:%M:%S") if dt is not None else "N/A"


In [25]:
THRESHOLD = 0.9
# K, N = 20, 30

K, N = 60, 80


# SPH = 5 * 60
# SOP = 60 * 60

SPH = 5 * 60
SOP = 30 * 60
REFRACTORY = SPH + SOP

BATCH_SIZE = 512
NORMALIZE = True
GAP_THRESHOLD_SECONDS = 300.0
EPS = 1e-6

In [26]:
records = []
seizures = set()

for path in TEST_DIR.rglob("*.npz"):
    with np.load(path, allow_pickle=False) as data:
        file_start = read_datetime(data["file_start_datetime"])
        sfreq = float(data["sfreq"])
        records.append((file_start, path))

        for start_sample, end_sample in data["seizure_intervals"]:
            seizures.add((
                file_start + timedelta(seconds=float(start_sample) / sfreq),
                file_start + timedelta(seconds=float(end_sample) / sfreq),
            ))

# Sắp xếp dữ liệu theo thời gian
if not records:
    raise ValueError(f"Không tìm thấy NPZ trong {TEST_DIR}")
records.sort(key=lambda item: item[0])
seizures = sorted(seizures)

votes = deque(maxlen=N)
blocked_until = None
previous_end = None

predictions = []
alarms = []
coverage = []
session_ready_times = []
session_window_count = 0
small_gaps = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

with torch.inference_mode():
    for file_start, path in records:
        with np.load(path, allow_pickle=False) as data:
            eeg = np.asarray(data["eeg"], dtype=np.float32)
            starts = np.asarray(data["start_sample"])
            stops = np.asarray(data["available_sample"])
            sfreq = float(data["sfreq"])
            if sfreq <= 0 or len(starts) != len(eeg) or len(stops) != len(eeg):
                raise ValueError(f"Metadata cửa sổ không hợp lệ: {path}")

        for first in range(0, len(eeg), BATCH_SIZE):
            last = min(first + BATCH_SIZE, len(eeg))

            x = torch.from_numpy(eeg[first:last]).to(device)
            if NORMALIZE:
                x = normalize_function(x, ea_matrix)
                # x = normalize_function(x, ea_matrix=None)

            _, logits = model(x)

            # Theo quy tắc nhãn: interictal=0, preictal=1.
            probabilities = logits.softmax(dim=-1)[:, 1].cpu().numpy()

            for index, probability in zip(range(first, last), probabilities):
                window_start = file_start + timedelta(seconds=float(starts[index] / sfreq))
                window_end = file_start + timedelta(seconds=float(stops[index] / sfreq))

                if window_end <= window_start:
                    raise ValueError(f"Cửa sổ có thời lượng không hợp lệ: {path}")

                # Kiểm tra thứ tự theo START: cho phép cửa sổ trượt chồng lấn.
                if predictions and window_start < predictions[-1]["window_start"]:
                    raise ValueError("Cửa sổ sai thứ tự thời gian; kiểm tra NPZ!")
                if previous_end is not None and window_end <= previous_end:
                    raise ValueError("Cửa sổ lặp hoặc thời điểm kết thúc không tăng!")

                gap = (
                    (window_start - previous_end).total_seconds()
                    if previous_end is not None else None
                )
                # Yêu cầu: gap < 300 s liên tục; đúng 300 s là phiên mới.
                new_session = gap is None or gap >= GAP_THRESHOLD_SECONDS

                if new_session:
                    votes.clear()
                    blocked_until = None
                    session_window_count = 0
                    coverage.append((window_start, window_end))
                    session_ready_times.append(None)
                else:
                    if gap > EPS:
                        small_gaps.append((previous_end, window_start))
                    coverage[-1] = (coverage[-1][0], window_end)

                session_window_count += 1
                if session_window_count == N:
                    # Dùng thời điểm cửa sổ thứ N thực sự kết thúc
                    session_ready_times[-1] = window_end

                # Điều kiện cần và đủ để phát alarm
                positive = int(probability >= THRESHOLD)
                votes.append(positive)
                positive_count = sum(votes)
                can_alarm = (blocked_until is None or window_end >= blocked_until)

                fire_alarm = (
                    len(votes) == N
                    and positive_count >= K
                    and can_alarm
                )

                if fire_alarm:
                    alarms.append({
                        "alarm_time": window_end,
                        "source_file": path.name,
                        "session_id": len(coverage),
                        "history_ready": len(votes) == N,
                        "sop_start": window_end + timedelta(seconds=SPH),
                        "sop_end": window_end + timedelta(seconds=SPH + SOP),
                        "probability": float(probability),
                        "positive_count": positive_count,
                    })

                    blocked_until = window_end + timedelta(seconds=REFRACTORY)
                    votes.clear()

                predictions.append({
                    "source_file": path.name,
                    "window_start": window_start,
                    "prediction_time": window_end,
                    "p_preictal": float(probability),
                    "positive": positive,
                    "positive_count": positive_count,
                    "alarm": int(fire_alarm),
                })

                previous_end = window_end


# ============================================================
# 1. THÔNG SỐ & HÀM XỬ LÝ KHOẢNG THỜI GIAN
# ============================================================

POST_HORIZON_SECONDS = 60 * 60
horizon_seconds = SPH + SOP

if SPH < 0 or SOP <= 0 or POST_HORIZON_SECONDS < 0:
    raise ValueError("SPH >= 0, SOP > 0 và POST_HORIZON_SECONDS >= 0.")


def merge_intervals(intervals):
    """Hợp các khoảng giao nhau để không cộng trùng thời gian."""
    intervals = sorted(
        (start, end)
        for start, end in intervals
        if start < end
    )

    merged = []
    for start, end in intervals:
        if not merged or start > merged[-1][1]:
            merged.append((start, end))
        else:
            merged[-1] = (
                merged[-1][0],
                max(merged[-1][1], end),
            )

    return merged


def subtract_intervals(base_intervals, excluded_intervals):
    """Lấy các khoảng base trừ đi các khoảng excluded."""
    excluded_intervals = merge_intervals(excluded_intervals)
    result = []

    for start, end in merge_intervals(base_intervals):
        cursor = start

        for ex_start, ex_end in excluded_intervals:
            if ex_end <= cursor:
                continue
            if ex_start >= end:
                break

            if cursor < ex_start:
                result.append((cursor, min(ex_start, end)))

            cursor = max(cursor, ex_end)
            if cursor >= end:
                break

        if cursor < end:
            result.append((cursor, end))

    return result


def duration_seconds(intervals):
    return sum(
        (end - start).total_seconds()
        for start, end in merge_intervals(intervals)
    )


seizures = sorted(seizures)
onsets = [start for start, end in seizures]


def get_session_id(time_point):
    """Xác định session_id (1-based) chứa mốc thời gian time_point."""
    for idx, (block_start, block_end) in enumerate(coverage, start=1):
        if block_start <= time_point <= block_end:
            return idx
    return None


def fully_covered(start, end):
    return any(
        block_start <= start and end <= block_end
        for block_start, block_end in coverage
    )


def containing_interval(time, intervals):
    return next(
        (
            (start, end)
            for start, end in intervals
            if start <= time < end
        ),
        None,
    )


# ============================================================
# 2. THỜI GIAN ĐỦ DỮ LIỆU ĐỂ ĐÁNH GIÁ (THEO TỪNG SESSION)
# ============================================================

evaluable_intervals = []
session_eval_seconds = {}

for idx, ((start, end), ready) in enumerate(zip(coverage, session_ready_times), start=1):
    if ready is None:
        session_eval_seconds[idx] = 0.0
        continue

    eval_start = ready
    eval_end = end - timedelta(seconds=horizon_seconds)

    if eval_start < eval_end:
        evaluable_intervals.append((eval_start, eval_end))
        session_eval_seconds[idx] = (eval_end - eval_start).total_seconds()
    else:
        session_eval_seconds[idx] = 0.0

evaluable_seconds = duration_seconds(evaluable_intervals)
evaluable_hours = evaluable_seconds / 3600


# ============================================================
# 3. NORMAL INTERVALS -> np
# ============================================================

excluded_intervals = [
    (
        onset - timedelta(seconds=horizon_seconds),
        seizure_end + timedelta(seconds=POST_HORIZON_SECONDS),
    )
    for onset, seizure_end in seizures
]

normal_intervals = subtract_intervals(
    evaluable_intervals,
    excluded_intervals,
)

np_seconds = duration_seconds(normal_intervals)
normal_hours = np_seconds / 3600


# ============================================================
# 4. PHÂN LOẠI ALARM & GHÉP ALARM CÙNG LOẠI TRONG HORIZON
# ============================================================

scored_alarms = []

for original_alarm in sorted(alarms, key=lambda alarm: alarm["alarm_time"]):
    alarm = dict(original_alarm)
    alarm_time = alarm["alarm_time"]

    alarm["sop_start"] = alarm_time + timedelta(seconds=SPH)
    alarm["sop_end"] = alarm_time + timedelta(seconds=horizon_seconds)

    history_ready = alarm.get("history_ready", False)

    matched_onsets = [
        onset
        for onset in onsets
        if alarm["sop_start"] <= onset < alarm["sop_end"]
    ]

    normal_interval = containing_interval(alarm_time, normal_intervals)

    if not history_ready:
        alarm["status"] = "unscorable"
    elif matched_onsets:
        alarm["status"] = "true_alarm"
    elif not fully_covered(alarm_time, alarm["sop_end"]):
        alarm["status"] = "unscorable"
    elif normal_interval is None:
        alarm["status"] = "excluded"
    else:
        alarm["status"] = "false_alarm"

    alarm["matched_onsets"] = matched_onsets
    scored_alarms.append(alarm)

grouped_alarms = []
active_group = {}

for alarm in scored_alarms:
    status = alarm["status"]

    if status not in ("true_alarm", "false_alarm"):
        continue

    previous = active_group.get(status)

    if (
        previous is not None
        and alarm["alarm_time"] < previous["sop_end"]
        and fully_covered(previous["alarm_time"], alarm["alarm_time"])
    ):
        previous["merged_count"] += 1
        previous["member_alarm_times"].append(alarm["alarm_time"])
        alarm["merged_into"] = previous["alarm_time"]
        continue

    group = dict(alarm)
    group["merged_count"] = 1
    group["member_alarm_times"] = [alarm["alarm_time"]]
    grouped_alarms.append(group)
    active_group[status] = group


# ============================================================
# 5. SENSITIVITY VÀ PREDICTION TIME
# ============================================================

seizure_results = []

for onset, seizure_end in seizures:
    required_start = onset - timedelta(seconds=horizon_seconds)
    eligible = any(
        ready is not None and ready <= required_start and onset <= end
        for (_, end), ready in zip(coverage, session_ready_times)
    )

    matched = [
        alarm
        for alarm in grouped_alarms
        if (
            alarm["status"] == "true_alarm"
            and alarm["sop_start"] <= onset < alarm["sop_end"]
        )
    ]

    first_alarm = min(
        matched,
        key=lambda alarm: alarm["alarm_time"],
        default=None,
    )

    seizure_results.append({
        "session_id": get_session_id(onset),
        "seizure_onset": onset,
        "seizure_end": seizure_end,
        "required_start": required_start,
        "eligible": eligible,
        "predicted": first_alarm is not None,
        "alarm_time": first_alarm["alarm_time"] if first_alarm else None,
        "lead_seconds": (
            (onset - first_alarm["alarm_time"]).total_seconds()
            if first_alarm else None
        ),
    })

eligible_seizures = [r for r in seizure_results if r["eligible"]]
predicted_count = sum(r["predicted"] for r in eligible_seizures)

sensitivity = (
    predicted_count / len(eligible_seizures)
    if eligible_seizures else None
)

prediction_times_seconds = [
    r["lead_seconds"]
    for r in eligible_seizures
    if r["predicted"] and r["lead_seconds"] is not None
]

mean_prediction_time_seconds = (
    mean(prediction_times_seconds)
    if prediction_times_seconds else None
)


# ============================================================
# 6. FALSE WAITING TIME (fwt) & ALARMS
# ============================================================

true_alarms = [
    alarm
    for alarm in grouped_alarms
    if alarm["status"] == "true_alarm"
]

false_alarms = [
    alarm
    for alarm in grouped_alarms
    if alarm["status"] == "false_alarm"
]

false_waiting_intervals = []
session_fwt_intervals = {idx: [] for idx in range(1, len(coverage) + 1)}

for alarm in false_alarms:
    normal_interval = containing_interval(alarm["alarm_time"], normal_intervals)
    if normal_interval is None:
        continue

    _, normal_end = normal_interval
    wait_start = alarm["alarm_time"]
    wait_end = min(alarm["sop_end"], normal_end)

    if wait_start < wait_end:
        false_waiting_intervals.append((wait_start, wait_end))
        sess_id = alarm.get("session_id") or get_session_id(wait_start)
        if sess_id in session_fwt_intervals:
            session_fwt_intervals[sess_id].append((wait_start, wait_end))

false_waiting_intervals = merge_intervals(false_waiting_intervals)
fwt_seconds = duration_seconds(false_waiting_intervals)

specificity = (
    1 - fwt_seconds / np_seconds
    if np_seconds > 0 else None
)

true_count = len(true_alarms)
false_count = len(false_alarms)
false_alarms_per_evaluable_hour = (
    false_count / evaluable_hours
    if evaluable_hours > 0 else None
)
false_alarms_per_normal_hour = (
    false_count / normal_hours
    if normal_hours > 0 else None
)


# ============================================================
# 7. HIỂN THỊ BẢNG TỔNG HỢP VỚI TABULATE
# ============================================================

session_rows = []

for sess_id in range(1, len(coverage) + 1):
    # Mốc dữ liệu của toàn phiên (có thể gồm nhiều file), khớp với visualize.
    # Đây không phải mốc bắt đầu/kết thúc của khoảng đủ điều kiện đánh giá.
    session_start, session_end = coverage[sess_id - 1]
    session_time_str = (
        f"{session_start:%Y-%m-%d %H:%M:%S}\n"
        f"{session_end:%Y-%m-%d %H:%M:%S}"
    )

    # 1. Thời lượng đánh giá của từng phiên liên tục (Session)
    eval_sec = session_eval_seconds.get(sess_id, 0.0)
    eval_duration_str = format_hms(eval_sec)

    # Thông báo riêng với phiên vẫn chạy inference nhưng không có khoảng đánh giá.
    ready = session_ready_times[sess_id - 1]
    if ready is None:
        evaluation_status = (
            f"Đã chạy inference\nChưa đủ {N} cửa sổ đầu tiên\n"
            "Không có khoảng đánh giá"
        )
    elif eval_sec <= 0:
        followup_seconds = max(0.0, (session_end - ready).total_seconds())
        evaluation_status = (
            "Đã chạy inference\nKhông đủ thời gian theo dõi\n"
            f"Sau đủ N cửa sổ: {format_hms(followup_seconds)}\n"
            f"Cần hơn SPH+SOP: {format_hms(horizon_seconds)}"
        )
    else:
        evaluation_status = "Đã chạy inference\nCó khoảng đủ điều kiện đánh giá"

    emitted_count = sum(
        (a.get("session_id") or get_session_id(a["alarm_time"])) == sess_id
        for a in alarms
    )
    unscorable_count = sum(
        (a.get("session_id") or get_session_id(a["alarm_time"])) == sess_id
        and a["status"] == "unscorable"
        for a in scored_alarms
    )
    evaluation_status += (
        f"\nAlarm đã phát: {emitted_count}\n"
        f"Alarm không thể đánh giá: {unscorable_count}"
    )

    # 2 & 3. Số cơn động kinh & Thời gian bắt đầu - kết thúc (hh:mm:ss)
    sess_seizures = [r for r in seizure_results if r["session_id"] == sess_id]
    seizure_count = len(sess_seizures)
    if sess_seizures:
        seizure_time_str = "\n".join(
            f"{format_time(r['seizure_onset'])} - {format_time(r['seizure_end'])}"
            for r in sess_seizures
        )
        # 6. Thời gian cảnh báo trước cơn (hh:mm:ss)
        lead_time_str = "\n".join(
            format_hms(r["lead_seconds"]) if r["lead_seconds"] is not None else "Không dự báo được"
            for r in sess_seizures
        )
    else:
        seizure_time_str = "-"
        lead_time_str = "-"

    # 4. Thời điểm phát cảnh báo đã hậu xử lý: gồm cả cảnh báo đúng và cảnh báo giả (hh:mm:ss)
    sess_alarms = [
        a for a in grouped_alarms
        if (a.get("session_id") or get_session_id(a["alarm_time"])) == sess_id
    ]
    if sess_alarms:
        alarm_times_str = "\n".join(
            f"{format_time(a['alarm_time'])} ({'Đúng' if a['status'] == 'true_alarm' else 'Giả'})"
            for a in sess_alarms
        )
    else:
        alarm_times_str = "-"

    # 5. Thời gian bệnh nhân phải ở trạng thái cảnh báo giả (hh:mm:ss)
    sess_fwt_sec = duration_seconds(session_fwt_intervals.get(sess_id, []))
    fwt_str = format_hms(sess_fwt_sec)

    session_rows.append([
        f"Session {sess_id}",
        session_time_str,
        eval_duration_str,
        evaluation_status,
        seizure_count,
        seizure_time_str,
        alarm_times_str,
        fwt_str,
        lead_time_str,
    ])

# Dòng tổng kết toàn bộ các phiên
session_rows.append([
    "Tổng / TB",
    "-",
    format_hms(evaluable_seconds),
    (
        f"Có khoảng đánh giá: "
        f"{sum(session_eval_seconds.get(sid, 0.0) > 0 for sid in range(1, len(coverage) + 1))}"
        f"/{len(coverage)} phiên"
    ),
    len(seizure_results),
    f"Đủ ĐK: {len(eligible_seizures)} | Dự báo đúng: {predicted_count}",
    f"Đúng: {true_count} | Giả: {false_count}",
    format_hms(fwt_seconds),
    format_hms(mean_prediction_time_seconds) if mean_prediction_time_seconds is not None else "N/A",
])

headers = [
    "Phiên",
    "Thời gian dữ liệu toàn phiên\n(Bắt đầu / Kết thúc)",
    "1. Thời lượng đánh giá\n(hh:mm:ss)",
    "Trạng thái đánh giá",
    "2. Số cơn\nđộng kinh",
    "3. Thời gian cơn động kinh\n(Bắt đầu - Kết thúc)",
    "4. Thời điểm phát\ncảnh báo (hh:mm:ss)",
    "5. Thời gian ở trạng thái\ncảnh báo giả (hh:mm:ss)",
    "6. Thời gian cảnh báo\ntrước cơn (hh:mm:ss)",
]

print("\n=== BẢNG KẾT QUẢ ĐÁNH GIÁ THEO PHIÊN (SESSION) ===")
print(tabulate(session_rows, headers=headers, tablefmt="grid", stralign="center", numalign="center"))

# Bảng chỉ số tổng hợp ngắn gọn
summary_metrics = [[
    f"{sensitivity * 100:.2f}%" if sensitivity is not None else "N/A",
    f"{specificity * 100:.2f}%" if specificity is not None else "N/A",
    f"{false_alarms_per_evaluable_hour:.4f}" if false_alarms_per_evaluable_hour is not None else "N/A",
    f"{false_alarms_per_normal_hour:.4f}" if false_alarms_per_normal_hour is not None else "N/A",
    format_hms(mean_prediction_time_seconds),
]]
summary_headers = [
    "Sensitivity",
    "Specificity (1 - fwt/np)",
    "FAR / Evaluable Hour",
    "FAR / Normal Hour",
    "Mean Lead Time (hh:mm:ss)",
]

print("\n=== CHỈ SỐ TỔNG HỢP ===")
print(tabulate(summary_metrics, headers=summary_headers, tablefmt="grid", stralign="center"))


=== BẢNG KẾT QUẢ ĐÁNH GIÁ THEO PHIÊN (SESSION) ===
+-----------+--------------------------------+--------------------------+---------------------------------+-------------+------------------------------+-----------------------+-----------------------------+-------------------------+
|   Phiên   |  Thời gian dữ liệu toàn phiên  |  1. Thời lượng đánh giá  |       Trạng thái đánh giá       |  2. Số cơn  |  3. Thời gian cơn động kinh  |   4. Thời điểm phát   |  5. Thời gian ở trạng thái  |  6. Thời gian cảnh báo  |
|           |      (Bắt đầu / Kết thúc)      |        (hh:mm:ss)        |                                 |  động kinh  |     (Bắt đầu - Kết thúc)     |  cảnh báo (hh:mm:ss)  |   cảnh báo giả (hh:mm:ss)   |  trước cơn (hh:mm:ss)   |
+===========+================================+==========================+=================================+=============+==============================+=======================+=============================+=========================+
| Session 1 |   

In [27]:
from datetime import timedelta as _TimelineDelta
from functools import lru_cache as _timeline_cache

import numpy as _tnp
import matplotlib.pyplot as _tplt
from matplotlib.patches import Patch as _TimelinePatch
from matplotlib.lines import Line2D as _TimelineLine
from matplotlib.ticker import FuncFormatter as _TimelineFormatter
import ipywidgets as _tw
from IPython.display import display as _timeline_display


# Postictal duration follows infer.py; it is a retrospective display convention.
TIMELINE_POST_SECONDS = float(globals().get("POST_HORIZON_SECONDS", 60 * 60))
TIMELINE_DEFAULT_VIEW_SECONDS = 30
TIMELINE_SPACING_UV = 400.0  # Twice the previous 200 µV channel spacing.
TIMELINE_FIGSIZE = (50, 24)  # Wider plot and twice the previous height.
TIMELINE_MAX_BINS = 1800    # Min/max envelope for long views; no change to NPZ.


def _timeline_union(intervals):
    merged = []
    for start, end in sorted(intervals):
        if start >= end:
            continue
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return merged


def _timeline_subtract(intervals, excluded):
    result = []
    excluded = _timeline_union(excluded)
    for start, end in _timeline_union(intervals):
        for left, right in excluded:
            if right <= start:
                continue
            if left >= end:
                break
            if start < left:
                result.append((start, left))
            start = max(start, right)
            if start >= end:
                break
        if start < end:
            result.append((start, end))
    return result


def _timeline_regions(events):
    """Annotated ictal and postictal after each seizure, matching infer.py.

    Ictal takes priority when a new seizure occurs during a postictal interval.
    These regions never determine the position or display of an alarm.
    """
    events = tuple(events)
    ictal = _timeline_union(events)
    post = _timeline_subtract([
        (end, end + _TimelineDelta(seconds=TIMELINE_POST_SECONDS))
        for _, end in events
    ], ictal)
    return {"Postictal (chú thích)": (post, "gold"),
            "Ictal (chú thích)": (ictal, "red")}


def _timeline_alarm_windows(emitted_alarms, sph_seconds, sop_seconds):
    """Snapshot every emitted alarm; no scoring, grouping or onset-based selection."""
    if not (_tnp.isfinite(sph_seconds) and sph_seconds >= 0
            and _tnp.isfinite(sop_seconds) and sop_seconds > 0):
        raise ValueError("SPH >= 0 and SOP > 0 are required.")
    result = []
    for original in emitted_alarms:
        time = original["alarm_time"]
        start = original.get("sop_start", time + _TimelineDelta(seconds=sph_seconds))
        end = original.get("sop_end", time + _TimelineDelta(seconds=sph_seconds+sop_seconds))
        if not time <= start < end:
            raise ValueError(f"Invalid SOP interval for alarm at {time}")
        result.append({"alarm_time": time, "sop_start": start, "sop_end": end,
                       "session_id": original.get("session_id")})
    return tuple(sorted(result, key=lambda alarm: alarm["alarm_time"]))


@_timeline_cache(maxsize=1)
def _timeline_load(path):
    # A compressed NPZ must decompress its eeg member. Keep only one file cached.
    with _tnp.load(path, allow_pickle=False) as data:
        signal = _tnp.asarray(data["eeg"], dtype=_tnp.float32)
        starts = _tnp.asarray(data["start_sample"], dtype=_tnp.int64)
        stops = _tnp.asarray(data["available_sample"], dtype=_tnp.int64)
        rate = float(data["sfreq"])
        names = tuple(str(name) for name in data["channels"].astype(str))
    if signal.ndim != 3 or signal.shape[1] != 18 or len(names) != 18:
        raise ValueError(f"Expected exactly 18 EEG channels: {path}")
    if (rate <= 0 or len(starts) != len(signal) or len(stops) != len(signal)
            or _tnp.any(stops - starts != signal.shape[2])
            or _tnp.any(_tnp.diff(starts) < 0)):
        raise ValueError(f"Invalid window metadata: {path}")
    return signal, starts, stops, rate, names


def _timeline_envelope(times, signal):
    """Retain chronological min/max per channel per bin, including narrow peaks."""
    step = max(1, int(_tnp.ceil(len(times) / TIMELINE_MAX_BINS)))
    if step == 1:
        return [(times, row) for row in signal]
    result = []
    for row in signal:
        indices = []
        for first in range(0, len(times), step):
            block = row[first:first + step]
            indices.extend(sorted({first + int(_tnp.argmin(block)),
                                   first + int(_tnp.argmax(block))}))
        indices = _tnp.asarray(indices)
        result.append((times[indices], row[indices]))
    return result


def show_eeg_timeline(inference_records, sessions, events, emitted_alarms,
                      sph_seconds, sop_seconds):
    """Display a session overview and all 18 channels in a movable time view."""
    if not inference_records or not sessions:
        raise ValueError("Run the infer.py cell first: records/coverage are empty.")
    if not (_tnp.isfinite(TIMELINE_POST_SECONDS) and TIMELINE_POST_SECONDS >= 0
            and TIMELINE_SPACING_UV > 0 and TIMELINE_MAX_BINS > 0):
        raise ValueError("Invalid timeline configuration.")
    regions = _timeline_regions(events)
    alarm_windows = _timeline_alarm_windows(emitted_alarms, sph_seconds, sop_seconds)
    # Snapshot inference metadata; do not change any inference variables.
    file_records = tuple(sorted(inference_records, key=lambda item: item[0]))
    sessions = tuple(sessions)
    session_select = _tw.Dropdown(
        options=[(f"Session {i}: {a:%Y-%m-%d %H:%M:%S} → {b:%Y-%m-%d %H:%M:%S}", i - 1)
                 for i, (a, b) in enumerate(sessions, 1)], description="Phiên:",
        layout=_tw.Layout(width="750px"))
    duration = _tw.Dropdown(options=[("10 giây", 10), ("30 giây", 30),
        ("1 phút", 60), ("5 phút", 300), ("30 phút", 1800),
        ("1 giờ", 3600), ("Toàn phiên", None)],
        value=TIMELINE_DEFAULT_VIEW_SECONDS, description="Đoạn xem:")
    offset = _tw.FloatSlider(value=0, min=0, max=1, step=1,
        description="Từ giây:", continuous_update=False,
        layout=_tw.Layout(width="750px"), readout_format=".0f")
    spacing = _tw.FloatLogSlider(value=TIMELINE_SPACING_UV, base=10,
        min=1, max=4, step=0.05, description="µV/kênh:", continuous_update=False)
    previous = _tw.Button(description="← Đoạn trước")
    following = _tw.Button(description="Đoạn sau →")
    show_sop = _tw.Checkbox(value=False, description="Hiện khoảng SOP sau alarm",
                            indent=False, layout=_tw.Layout(width="300px"))
    output = _tw.Output()
    state = {"updating": False}

    def render(*_):
        if state["updating"]:
            return
        session_start, session_end = sessions[session_select.value]
        total = (session_end - session_start).total_seconds()
        width = min(float(duration.value or total), total)
        left = session_start + _TimelineDelta(seconds=offset.value)
        right = min(left + _TimelineDelta(seconds=width), session_end)
        session_alarms = [alarm for alarm in alarm_windows
            if (alarm["session_id"] == session_select.value + 1
                if alarm["session_id"] is not None
                else session_start <= alarm["alarm_time"] <= session_end)]
        with output:
            output.clear_output(wait=True)
            alarm_times_text = "; ".join(
                alarm["alarm_time"].strftime("%Y-%m-%d %H:%M:%S")
                for alarm in session_alarms
            ) or "Không có alarm"
            _timeline_display(_tw.HTML(
                f"<b>Thời gian phát cảnh báo trong phiên:</b> {alarm_times_text}"
            ))
            fig, (overview, ax) = _tplt.subplots(2, 1, figsize=TIMELINE_FIGSIZE,
                gridspec_kw={"height_ratios": [1, 12]}, constrained_layout=True)
            try:
                # Overview covers the complete session; the outlined box is the view.
                for axis in (overview, ax):
                    for intervals, color in regions.values():
                        for a, b in intervals:
                            a, b = max(a, session_start), min(b, session_end)
                            if a < b:
                                axis.axvspan((a-session_start).total_seconds(),
                                    (b-session_start).total_seconds(),
                                    color=color, alpha=0.23, linewidth=0)
                    for alarm in session_alarms:
                        time = alarm["alarm_time"]
                        if show_sop.value:
                            a = max(alarm["sop_start"], session_start)
                            b = min(alarm["sop_end"], session_end)
                            if a < b:
                                axis.axvspan((a-session_start).total_seconds(),
                                    (b-session_start).total_seconds(),
                                    facecolor="none", edgecolor="green", hatch="//",
                                    alpha=0.35, linewidth=0.7, zorder=2)
                        if axis is overview and session_start <= time <= session_end:
                            axis.axvline((time-session_start).total_seconds(),
                                color="green", linewidth=1.3, linestyle="--", zorder=5)
                overview.axvspan(offset.value, (right-session_start).total_seconds(),
                    facecolor="none", edgecolor="black", linewidth=1.5)
                overview.set(xlim=(0, total), yticks=[],
                    title=f"Toàn bộ phiên | {len(session_alarms)} alarm đã phát | khung đen: đoạn đang xem")
                overview.title.set_fontsize(35)
                overview.set_facecolor("#f5f5f5")
                offsets = _tnp.arange(17, -1, -1) * spacing.value
                signal_min, signal_max = float(offsets[-1]), float(offsets[0])
                channel_names = None
                last_drawn_end = None
                for file_start, path in file_records:
                    if file_start >= right:
                        break
                    # Skip unrelated archives without decompressing EEG.
                    with _tnp.load(path, allow_pickle=False) as metadata:
                        file_stops = metadata["available_sample"]
                        file_rate = float(metadata["sfreq"])
                        if not len(file_stops):
                            continue
                        file_end = file_start + _TimelineDelta(seconds=float(file_stops[-1])/file_rate)
                    if file_end <= left:
                        continue
                    signal, starts, stops, rate, names = _timeline_load(path)
                    if channel_names is not None and names != channel_names:
                        raise ValueError(f"Channel order differs across files: {path}")
                    channel_names = names
                    lo = (left-file_start).total_seconds() * rate
                    hi = (right-file_start).total_seconds() * rate
                    selected = _tnp.flatnonzero((starts < hi) & (stops > lo))
                    if not len(selected):
                        continue
                    # Join touching windows, trim overlapping windows, leave real gaps blank.
                    runs = []
                    for index in selected:
                        first = max(int(starts[index]), int(_tnp.ceil(lo - 1e-6)))
                        end = min(int(stops[index]), int(_tnp.ceil(hi - 1e-6)))
                        absolute_first = file_start + _TimelineDelta(seconds=first/rate)
                        if last_drawn_end is not None and absolute_first < last_drawn_end:
                            first = max(first, int(round((last_drawn_end-file_start).total_seconds()*rate)))
                        if first >= end:
                            continue
                        values = signal[index, :, first-starts[index]:end-starts[index]]
                        if runs and first == runs[-1][1]:
                            runs[-1][1] = end
                            runs[-1][2].append(values)
                        else:
                            runs.append([first, end, [values]])
                        last_drawn_end = file_start + _TimelineDelta(seconds=end/rate)
                    for first, end, pieces in runs:
                        values = _tnp.concatenate(pieces, axis=1) * 1e6  # NPZ units: V -> µV.
                        times = (file_start-session_start).total_seconds() + _tnp.arange(first, end)/rate
                        for ch, (x, y) in enumerate(_timeline_envelope(times, values)):
                            shifted = y + offsets[ch]
                            finite = shifted[_tnp.isfinite(shifted)]
                            if finite.size:
                                signal_min = min(signal_min, float(finite.min()))
                                signal_max = max(signal_max, float(finite.max()))
                            ax.plot(x, shifted, color="#202b38", linewidth=0.55)
                if channel_names is None:
                    ax.text(0.5, 0.5, "Không có dữ liệu EEG trong đoạn này",
                            transform=ax.transAxes, ha="center")
                    channel_names = tuple(f"Kênh {i}" for i in range(1, 19))
                ax.set_yticks(offsets, channel_names)
                ax.tick_params(axis="both", labelsize=28)
                overview.tick_params(axis="x", labelsize=24)
                # Include every visible peak, with extra space for alarm labels above FP1-F7.
                ax.set_ylim(signal_min - spacing.value,
                            signal_max + 2 * spacing.value)
                ax.set_xlim(offset.value, (right-session_start).total_seconds())
                visible_alarms = [alarm for alarm in session_alarms
                                  if left <= alarm["alarm_time"] <= right]
                for index, alarm in enumerate(visible_alarms):
                    time = alarm["alarm_time"]
                    x = (time-session_start).total_seconds()
                    # Draw after EEG: a clear alarm boundary across all 18 channels.
                    ax.axvline(x, ymin=0, ymax=1, color="green",
                        linewidth=2.4, linestyle=(0, (6, 4)), alpha=1.0,
                        zorder=10, clip_on=False)
                    near_right = (time-left).total_seconds() > 0.85 * (right-left).total_seconds()
                    
                    ax.annotate(f"{time:%H:%M:%S}",
                        xy=(x, 0.99 - 0.025 * (index % 3)),
                        xycoords=ax.get_xaxis_transform(),
                        xytext=(-4 if near_right else 4, -2), textcoords="offset points",
                        ha="right" if near_right else "left", va="top", color="green",
                        fontsize=36, zorder=11,
                        bbox={"facecolor": "white", "edgecolor": "green", "alpha": 0.85,
                              "boxstyle": "round,pad=0.25"})
                # ax.set_title(f"EEG 18 kênh | {left:%Y-%m-%d %H:%M:%S} → {right:%Y-%m-%d %H:%M:%S}")
                ax.set_title(
                    f"Diễn biến EEG và cảnh báo theo thời gian\n"
                    f"{left:%Y-%m-%d %H:%M:%S} → {right:%Y-%m-%d %H:%M:%S}",
                    fontsize=40
                )
                # ax.set_xlabel("Thời gian ghi (ngày và giờ); vùng trống là khoảng không có dữ liệu", fontsize=30)
                # ax.set_ylabel(f"Khoảng cách kênh: {spacing.value:.0f} µV",  fontsize=30)
                ax.grid(axis="y", alpha=0.2)
                for axis in (overview, ax):
                    axis.xaxis.set_major_formatter(_TimelineFormatter(
                        lambda seconds, _: (session_start + _TimelineDelta(seconds=seconds)).strftime("%d/%m\n%H:%M:%S")))
                legend = [_TimelinePatch(facecolor=color, alpha=0.35, label=name)
                    for name, (_, color) in regions.items()]
                legend.append(_TimelineLine([0], [0], color="green",
                    linewidth=2.4, linestyle=(0, (6, 4)),
                    label="Alarm đã phát"))
                if show_sop.value:
                    legend.append(_TimelinePatch(facecolor="none", edgecolor="green",
                        hatch="//", label="Khoảng dự báo SOP sau alarm"))
                ax.legend(handles=legend, loc="upper center",
                          bbox_to_anchor=(0.5, -0.065), ncol=4, fontsize=28)

                save_dir = Path("/kaggle/working/eeg_figures")
                save_dir.mkdir(parents=True, exist_ok=True)
                
                image_path = save_dir / (
                    f"session_{session_select.value + 1}_"
                    f"{left:%Y%m%d_%H%M%S}_{right:%Y%m%d_%H%M%S}.png"
                )
                
                fig.savefig(image_path, dpi=200, bbox_inches="tight", facecolor="white")
                print(f"Đã lưu ảnh: {image_path}")
                _tplt.show()
            finally:
                _tplt.close(fig)

    def update_range(change=None):
        state["updating"] = True
        try:
            a, b = sessions[session_select.value]
            total = (b-a).total_seconds()
            offset.max = max(0, total - min(float(duration.value or total), total))
            offset.value = 0 if change and change["owner"] is session_select else min(offset.value, offset.max)
            offset.disabled = offset.max == 0
        finally:
            state["updating"] = False
        render()

    def move(direction):
        a, b = sessions[session_select.value]
        step = float(duration.value or (b-a).total_seconds())
        offset.value = min(offset.max, max(0, offset.value + direction*step))

    session_select.observe(update_range, names="value")
    duration.observe(update_range, names="value")
    offset.observe(render, names="value")
    spacing.observe(render, names="value")
    show_sop.observe(render, names="value")
    previous.on_click(lambda _: move(-1))
    following.on_click(lambda _: move(1))
    _timeline_display(_tw.VBox([session_select, _tw.HBox([duration, spacing]),
        show_sop, _tw.HTML("Vạch xanh: alarm từ hệ thống. Đỏ/vàng: chú thích hồi cứu. "
            "SOP: khoảng dự báo sau alarm, bắt đầu sau SPH; không phải nhãn pre-ictal."),
        offset, _tw.HBox([previous, following]), output]))
    update_range()
    return {"session": session_select, "duration": duration,
            "offset": offset, "spacing": spacing, "show_sop": show_sop, "output": output}


if not all(name in globals() for name in ("records", "coverage", "seizures", "alarms", "SPH", "SOP")):
    raise RuntimeError("Hãy chạy cell infer.py trước, rồi chạy cell trực quan hóa này.")

eeg_timeline_viewer = show_eeg_timeline(records, coverage, seizures, alarms, SPH, SOP)
